# Sistema Quantitativo para Previsão de Demanda e Planejamento da Produção Automotiva

## Projeto demonstrativo para entrevista — Stellantis (v2: validação estatística ampliada)

Este notebook constrói um fluxo analítico completo que conecta **dados públicos**, **previsão de demanda** e **otimização de produção**. A análise usa a série mensal pública **Total Vehicle Sales (TOTALSA)**, cujo dado original é do U.S. Bureau of Economic Analysis e é disponibilizado pelo FRED. A série é agregada para o mercado norte-americano e está em taxa anual ajustada sazonalmente (SAAR). [Fonte FRED](https://fred.stlouisfed.org/series/TOTALSA)

> **Escopo e integridade:** este é um caso didático. Ele não usa dados internos, capacidade real, mix de produtos, metas, custos ou decisões da Stellantis. Os parâmetros da etapa de otimização são hipóteses explicitamente identificadas e editáveis.

### Pergunta de negócio

Como transformar uma previsão mensal de vendas do mercado em um plano de produção que equilibre nível de serviço, estoque e restrição de capacidade — e como comunicar, de forma defensável, o quanto se pode confiar nessa previsão?

### O que esta versão adiciona em relação a um protótipo simples

| Etapa | Entrega |
|---|---|
| Qualidade de dados | Checagem de lacunas, duplicidades e outliers (IQR) |
| Diagnóstico | Estacionariedade (teste ADF), decomposição STL, ACF/PACF |
| Validação | Backtest **walk-forward** (janela expansiva, 4 dobras) comparando 3 modelos |
| Diagnóstico de resíduos | Teste de Ljung-Box e ACF dos erros do modelo vencedor |
| Forecasting | Previsão para os seis meses seguintes, reajustada com toda a série |
| Incerteza | Cenários via **bootstrap** dos erros observados no backtest (sem assumir normalidade) |
| Otimização | Plano de produção avaliado em **três cenários** de demanda |
| Sensibilidade | Mapa de sensibilidade da decisão a capacidade e participação de mercado |

Uma validação em um único corte treino/teste mede o erro em apenas um período, que pode ser atipicamente fácil ou difícil de prever. Um teste de robustez natural é observar se a conclusão se sustenta em múltiplas janelas temporais — é isso que o backtest walk-forward abaixo faz.


In [ ]:
# Instala dependências caso o notebook esteja em um ambiente limpo, como o Google Colab.
%pip -q install statsmodels pulp scikit-learn


In [ ]:
# Bibliotecas e configuração
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.linear_model import Ridge
import pulp

# Estilo visual: paleta compatível com daltonismo (azul/laranja como par primário)
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "figure.dpi": 110,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})
COR_PRIMARIA = "#1f4e79"
COR_SECUNDARIA = "#ed7d31"
COR_TERCIARIA = "#2f75b5"
COR_ALERTA = "#c00000"
sns.set_palette([COR_PRIMARIA, COR_SECUNDARIA, "#70ad47", "#8172b3"])
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=TOTALSA"
FRED_SERIES_URL = "https://fred.stlouisfed.org/series/TOTALSA"

# Parâmetros de validação e previsão
N_DOBRAS_CV = 4                                      # janelas de backtest walk-forward
TAMANHO_DOBRA = 6                                     # meses por janela de teste
HORIZONTE_TESTE_TOTAL = N_DOBRAS_CV * TAMANHO_DOBRA   # 24 meses reservados no total
HORIZONTE_PREVISAO = 6                                # meses de planejamento futuro
N_BOOTSTRAP = 2000                                    # réplicas para o intervalo de incerteza
SEED = 42
rng = np.random.default_rng(SEED)

print("Configuração concluída.")


## 1. Coleta e qualidade dos dados

A série é carregada diretamente da fonte pública, o que torna a análise reproduzível. Como o TOTALSA é uma taxa anual ajustada sazonalmente, divide-se o valor por 12 apenas para criar uma **aproximação de unidades mensais** que facilite as visualizações; o modelo é treinado sobre a própria série oficial.

Antes de qualquer modelagem, a base passa por três checagens: continuidade mensal, duplicidades e outliers. Em seguida, um teste de estacionariedade e uma decomposição STL orientam a escolha da família de modelos.


In [ ]:
# Carregamento robusto da série oficial do FRED, com tratamento explícito de falha de rede
try:
    raw = pd.read_csv(FRED_CSV_URL)
except Exception as erro:
    raise RuntimeError(
        "Não foi possível baixar a série TOTALSA do FRED. Verifique a conexão de "
        "internet do ambiente (o Colab exige rede ativa durante a execução). "
        f"Detalhe técnico: {erro}"
    ) from erro

colunas_esperadas = {"observation_date", "TOTALSA"}
if not colunas_esperadas.issubset(raw.columns):
    raise ValueError(f"Formato inesperado. Colunas encontradas: {list(raw.columns)}")

duplicidades_brutas = raw["observation_date"].duplicated().sum()

df = (raw.rename(columns={"observation_date": "data", "TOTALSA": "vendas_saar_milhoes"})
        .assign(data=lambda x: pd.to_datetime(x["data"], errors="coerce"),
                vendas_saar_milhoes=lambda x: pd.to_numeric(x["vendas_saar_milhoes"], errors="coerce"))
        .dropna(subset=["data", "vendas_saar_milhoes"])
        .sort_values("data")
        .drop_duplicates("data")
        .reset_index(drop=True))

# Aproximação mensal: valor SAAR / 12. Mantida apenas para leitura operacional.
df["demanda_mensal_est_milhoes"] = df["vendas_saar_milhoes"] / 12
df["mes"] = df["data"].dt.month
df["ano"] = df["data"].dt.year
df["variacao_mensal_pct"] = df["vendas_saar_milhoes"].pct_change() * 100
df["variacao_anual_pct"] = df["vendas_saar_milhoes"].pct_change(12) * 100

# Continuidade mensal: cada intervalo entre observações deve ter entre 28 e 31 dias
intervalos_dias = df["data"].diff().dt.days.dropna()
meses_irregulares = (~intervalos_dias.between(28, 31)).sum()

print(f"Período disponível: {df['data'].min():%m/%Y} a {df['data'].max():%m/%Y}")
print(f"Observações: {len(df):,}")
print(f"Duplicidades de data no arquivo bruto: {duplicidades_brutas}")
print(f"Valores ausentes após limpeza: {df[['data', 'vendas_saar_milhoes']].isna().sum().sum()}")
print(f"Meses com intervalo irregular: {meses_irregulares}")

df.head()


In [ ]:
# Verificação de outliers (método IQR) — diagnóstico, sem remoção automática
q1, q3 = df["vendas_saar_milhoes"].quantile([0.25, 0.75])
iqr = q3 - q1
limite_inferior, limite_superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["vendas_saar_milhoes"] < limite_inferior) | (df["vendas_saar_milhoes"] > limite_superior)]

print(f"Faixa esperada (IQR x1.5): [{limite_inferior:.2f}, {limite_superior:.2f}] milhões SAAR")
print(f"Observações fora da faixa: {len(outliers)} de {len(df)} ({len(outliers) / len(df) * 100:.1f}%)")
if len(outliers) > 0:
    display(outliers[["data", "vendas_saar_milhoes"]])
print(
    "Esses pontos não foram removidos: em uma série de mercado, valores extremos costumam "
    "refletir choques reais (ex.: recessões, pandemia) e não erros de coleta. Descartá-los "
    "distorceria o diagnóstico de risco em vez de corrigir um problema de dados."
)


In [ ]:
# Teste de estacionariedade (Dickey-Fuller aumentado)
adf_nivel = adfuller(df["vendas_saar_milhoes"].dropna(), autolag="AIC")
adf_diferenca = adfuller(df["vendas_saar_milhoes"].diff().dropna(), autolag="AIC")

print("ADF em nível:")
print(f"  estatística = {adf_nivel[0]:.3f} | p-valor = {adf_nivel[1]:.4f}")
print("ADF em primeira diferença:")
print(f"  estatística = {adf_diferenca[0]:.3f} | p-valor = {adf_diferenca[1]:.4f}")

if adf_nivel[1] > 0.05:
    print(
        f"\nA série em nível não rejeita a hipótese nula de raiz unitária (p-valor "
        f"{adf_nivel[1]:.3f} > 0.05): há evidência de não estacionariedade. Isso justifica "
        "usar modelos com componente de tendência e sazonalidade explícitos (Holt-Winters) "
        "em vez de métodos que assumem média constante."
    )
else:
    print(f"\nA série em nível rejeita a hipótese nula de raiz unitária (p-valor {adf_nivel[1]:.3f} ≤ 0.05).")


In [ ]:
# Decomposição STL: separa tendência, sazonalidade e resíduo
stl_resultado = STL(df.set_index("data")["vendas_saar_milhoes"], period=12, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
axes[0].plot(stl_resultado.observed, color=COR_PRIMARIA, linewidth=1.4)
axes[0].set_title("Série observada", fontweight="bold", loc="left")
axes[1].plot(stl_resultado.trend, color=COR_SECUNDARIA, linewidth=1.6)
axes[1].set_title("Tendência", loc="left")
axes[2].plot(stl_resultado.seasonal, color=COR_TERCIARIA, linewidth=1.0)
axes[2].set_title("Sazonalidade", loc="left")
axes[3].plot(stl_resultado.resid, color="gray", linewidth=0.9)
axes[3].axhline(0, color="black", linewidth=0.6)
axes[3].set_title("Resíduo", loc="left")
fig.suptitle("Decomposição STL — TOTALSA", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Perfil sazonal médio e variação anual
nomes_meses = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
perfil_sazonal = df.groupby("mes", as_index=False)["vendas_saar_milhoes"].mean()
perfil_sazonal["nome_mes"] = nomes_meses

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=perfil_sazonal, x="nome_mes", y="vendas_saar_milhoes", color=COR_TERCIARIA, ax=axes[0])
axes[0].set_title("Perfil médio de sazonalidade", fontweight="bold")
axes[0].set_xlabel("Mês")
axes[0].set_ylabel("Milhões de unidades (SAAR)")

axes[1].bar(df["data"], df["variacao_anual_pct"], color=COR_PRIMARIA, width=22)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Variação anual (YoY)", fontweight="bold")
axes[1].set_ylabel("% vs. mesmo mês do ano anterior")
plt.tight_layout()
plt.show()

perfil_sazonal[["nome_mes", "vendas_saar_milhoes"]]


In [ ]:
# Autocorrelação e autocorrelação parcial — orientam a escolha de defasagens e de seasonal_periods
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df["vendas_saar_milhoes"], lags=36, ax=axes[0], color=COR_PRIMARIA)
axes[0].set_title("Função de autocorrelação (ACF)", fontweight="bold")
plot_pacf(df["vendas_saar_milhoes"], lags=36, ax=axes[1], method="ywm", color=COR_SECUNDARIA)
axes[1].set_title("Função de autocorrelação parcial (PACF)", fontweight="bold")
plt.tight_layout()
plt.show()


## 2. Backtest walk-forward e seleção de modelo

Um único corte treino/teste mede o erro em apenas um período — que pode ser atipicamente fácil ou difícil de prever. Para tornar a validação mais robusta, esta versão usa **validação cruzada temporal com janela expansiva** (*walk-forward*, *rolling-origin*): o modelo é reestimado em cada uma de `N_DOBRAS_CV` janelas sucessivas, sempre treinando apenas com dados anteriores ao período testado — nunca embaralhando o tempo.

Três modelos são comparados:

1. **Referência sazonal ingênua** — média histórica do mesmo mês, só com dados de treino.
2. **Holt-Winters aditivo** — nível, tendência e sazonalidade de 12 meses.
3. **Regressão com defasagens (Ridge)** — defasagens de 1 e 12 meses, tendência linear e dummies de mês, com previsão recursiva multi-passo (cada passo usa a previsão do passo anterior, como aconteceria em um cenário real de produção).

O objetivo não é escolher o modelo mais sofisticado, mas o que entrega o menor erro fora da amostra, de forma consistente entre janelas.


In [ ]:
def metricas(y_real, y_previsto):
    """Calcula MAE, RMSE e MAPE entre valores reais e previstos."""
    y_real = np.asarray(y_real, dtype=float)
    y_previsto = np.asarray(y_previsto, dtype=float)
    mae = np.mean(np.abs(y_real - y_previsto))
    rmse = np.sqrt(np.mean((y_real - y_previsto) ** 2))
    mape = np.mean(np.abs((y_real - y_previsto) / y_real)) * 100
    return {"MAE (milhões SAAR)": mae, "RMSE (milhões SAAR)": rmse, "MAPE (%)": mape}


def construir_dobras(dados, n_dobras=N_DOBRAS_CV, tamanho_dobra=TAMANHO_DOBRA):
    """Gera pares (índice de treino, índice de teste) em janela expansiva (walk-forward)."""
    dobras = []
    n = len(dados)
    inicio_teste = n - n_dobras * tamanho_dobra
    for k in range(n_dobras):
        fim_treino = inicio_teste + k * tamanho_dobra
        fim_teste = fim_treino + tamanho_dobra
        dobras.append((slice(0, fim_treino), slice(fim_treino, fim_teste)))
    return dobras


def prever_sazonal_naive(treino, datas_teste):
    """Referência simples: média histórica do mesmo mês, estimada apenas no treino."""
    medias = treino.groupby(treino["data"].dt.month)["vendas_saar_milhoes"].mean()
    return np.array([medias[d.month] for d in datas_teste])


def prever_holt_winters(treino, n_periodos):
    """Holt-Winters aditivo: nível, tendência e sazonalidade de 12 meses."""
    modelo = ExponentialSmoothing(
        treino["vendas_saar_milhoes"], trend="add", seasonal="add",
        seasonal_periods=12, initialization_method="estimated"
    ).fit(optimized=True)
    return np.asarray(modelo.forecast(n_periodos))


COLUNAS_X_REGRESSAO = ["lag_1", "lag_12", "tendencia"] + [f"mes_{m}" for m in range(2, 13)]

def construir_features_regressao(dados):
    """Cria defasagens (t-1, t-12), tendência linear e dummies de mês."""
    feats = dados.copy()
    feats["lag_1"] = feats["vendas_saar_milhoes"].shift(1)
    feats["lag_12"] = feats["vendas_saar_milhoes"].shift(12)
    feats["tendencia"] = np.arange(len(feats))
    dummies = pd.get_dummies(feats["mes"], prefix="mes", drop_first=True).astype(float)
    return pd.concat([feats, dummies], axis=1)


def prever_regressao_defasagens(treino, n_periodos, colunas_x=COLUNAS_X_REGRESSAO):
    """Regressão Ridge com defasagens; previsão recursiva multi-passo (cada passo usa a
    própria previsão do passo anterior como lag_1, já que o valor futuro real não é
    conhecido de antemão em um cenário real de produção)."""
    treino_feats = construir_features_regressao(treino).dropna(subset=colunas_x + ["vendas_saar_milhoes"])
    modelo = Ridge(alpha=1.0)
    modelo.fit(treino_feats[colunas_x], treino_feats["vendas_saar_milhoes"])

    historico = treino[["data", "vendas_saar_milhoes", "mes"]].copy()
    previsoes = []
    for _ in range(n_periodos):
        proxima_data = historico["data"].max() + pd.offsets.MonthBegin(1)
        linha = {
            "lag_1": historico["vendas_saar_milhoes"].iloc[-1],
            "lag_12": historico["vendas_saar_milhoes"].iloc[-12],
            "tendencia": len(historico),
        }
        for m in range(2, 13):
            linha[f"mes_{m}"] = 1.0 if proxima_data.month == m else 0.0
        x_novo = pd.DataFrame([linha])[colunas_x]
        previsao = float(modelo.predict(x_novo)[0])
        previsoes.append(previsao)
        nova_linha = pd.DataFrame({
            "data": [proxima_data], "vendas_saar_milhoes": [previsao], "mes": [proxima_data.month]
        })
        historico = pd.concat([historico, nova_linha], ignore_index=True)
    return np.array(previsoes)


print("Funções de modelagem e validação definidas.")


In [ ]:
# Execução do backtest walk-forward
dobras = construir_dobras(df)
registros = []
previsoes_por_modelo = {"Referência sazonal": [], "Holt-Winters": [], "Regressão com defasagens": []}
reais_por_dobra = []

for k, (idx_treino, idx_teste) in enumerate(dobras, start=1):
    treino_k = df.iloc[idx_treino].reset_index(drop=True)
    teste_k = df.iloc[idx_teste].reset_index(drop=True)
    periodo = f"{teste_k['data'].min():%m/%Y}–{teste_k['data'].max():%m/%Y}"
    reais_por_dobra.append(teste_k["vendas_saar_milhoes"].to_numpy())

    previsoes_k = {
        "Referência sazonal": prever_sazonal_naive(treino_k, teste_k["data"]),
        "Holt-Winters": prever_holt_winters(treino_k, len(teste_k)),
        "Regressão com defasagens": prever_regressao_defasagens(treino_k, len(teste_k)),
    }
    for nome_modelo, previsto in previsoes_k.items():
        previsoes_por_modelo[nome_modelo].append(previsto)
        m = metricas(teste_k["vendas_saar_milhoes"], previsto)
        registros.append({"dobra": k, "período": periodo, "modelo": nome_modelo, **m})

resultados_cv = pd.DataFrame(registros)
display(resultados_cv.style.format({
    "MAE (milhões SAAR)": "{:.3f}", "RMSE (milhões SAAR)": "{:.3f}", "MAPE (%)": "{:.2f}%"
}))


In [ ]:
# Agregação por modelo: média e desvio-padrão do erro entre as dobras
resumo_cv = (resultados_cv.groupby("modelo")["MAPE (%)"]
             .agg(mape_medio="mean", mape_desvio="std")
             .sort_values("mape_medio")
             .reset_index())
display(resumo_cv.style.format({"mape_medio": "{:.2f}%", "mape_desvio": "{:.2f}%"}))

melhor_modelo = resumo_cv.loc[0, "modelo"]
print(f"Modelo selecionado pela menor MAPE média nas {N_DOBRAS_CV} dobras: {melhor_modelo}")
print(
    f"MAPE médio: {resumo_cv.loc[0, 'mape_medio']:.2f}% (desvio-padrão entre dobras: "
    f"{resumo_cv.loc[0, 'mape_desvio']:.2f} p.p.). Com apenas {N_DOBRAS_CV} dobras, essa "
    "comparação deve ser lida como direcional — mais histórico ou reamostragem seriam "
    "necessários para um teste estatístico formal de diferença entre modelos."
)

fig, ax = plt.subplots(figsize=(10, 5))
cores_barras = [COR_PRIMARIA if m == melhor_modelo else "#b0b0b0" for m in resumo_cv["modelo"]]
ax.bar(resumo_cv["modelo"], resumo_cv["mape_medio"], yerr=resumo_cv["mape_desvio"], capsize=5, color=cores_barras)
ax.set_title("MAPE médio por modelo (± desvio-padrão entre dobras)", fontweight="bold")
ax.set_ylabel("MAPE (%)")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()


In [ ]:
# Diagnóstico de resíduos do modelo vencedor (previsões fora da amostra agrupadas das 4 dobras)
reais_vencedor = np.concatenate(reais_por_dobra)
previsoes_vencedor = np.concatenate(previsoes_por_modelo[melhor_modelo])
residuos_pool = reais_vencedor - previsoes_vencedor

teste_ljung_box = acorr_ljungbox(residuos_pool, lags=[6, 12], return_df=True)
display(teste_ljung_box)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(residuos_pool, lags=11, ax=axes[0], color=COR_PRIMARIA)
axes[0].set_title("ACF dos resíduos (fora da amostra)", fontweight="bold")
axes[1].hist(residuos_pool, bins=10, color=COR_TERCIARIA, edgecolor="white")
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Distribuição dos resíduos", fontweight="bold")
plt.tight_layout()
plt.show()

print(
    f"Resíduos agrupados das {N_DOBRAS_CV} dobras (n={len(residuos_pool)}). Com uma amostra "
    "pequena, o teste de Ljung-Box tem poder estatístico limitado; ele é reportado como "
    "evidência complementar, não como prova definitiva de ausência de autocorrelação."
)


## 3. Previsão de seis meses e incerteza via bootstrap

O modelo vencedor é reajustado com toda a série disponível. Em vez de assumir que os erros seguem uma distribuição normal, a incerteza é estimada por **bootstrap dos resíduos observados no backtest walk-forward**: os erros históricos são reamostrados com reposição e somados à previsão pontual, gerando uma distribuição empírica da demanda futura. Essa abordagem não exige a suposição de normalidade e captura melhor caudas assimétricas — uma escolha deliberadamente mais conservadora do que aproximar o erro por ±k desvios-padrão.


In [ ]:
# Reajuste do modelo vencedor com toda a série histórica
data_inicial_futura = df["data"].max() + pd.offsets.MonthBegin(1)
datas_futuras = pd.date_range(data_inicial_futura, periods=HORIZONTE_PREVISAO, freq="MS")

if melhor_modelo == "Holt-Winters":
    modelo_final = ExponentialSmoothing(
        df["vendas_saar_milhoes"], trend="add", seasonal="add",
        seasonal_periods=12, initialization_method="estimated"
    ).fit(optimized=True)
    previsao_pontual = np.asarray(modelo_final.forecast(HORIZONTE_PREVISAO))
elif melhor_modelo == "Regressão com defasagens":
    previsao_pontual = prever_regressao_defasagens(df, HORIZONTE_PREVISAO)
else:
    previsao_pontual = prever_sazonal_naive(df, datas_futuras)

# Bootstrap dos resíduos do backtest para gerar cenários empíricos (p10 / p50 / p90)
amostras_bootstrap = rng.choice(residuos_pool, size=(N_BOOTSTRAP, HORIZONTE_PREVISAO), replace=True)
simulacoes = previsao_pontual[None, :] + amostras_bootstrap

previsoes = pd.DataFrame({
    "data": datas_futuras,
    "cenario_conservador": np.maximum(np.percentile(simulacoes, 10, axis=0), 0),
    "cenario_base": previsao_pontual,
    "cenario_otimista": np.percentile(simulacoes, 90, axis=0),
})
previsoes["demanda_mensal_base_milhoes"] = previsoes["cenario_base"] / 12

print(f"Modelo reajustado: {melhor_modelo}")
print(f"Réplicas de bootstrap: {N_BOOTSTRAP:,} | faixa reportada: p10-p90")
display(previsoes.style.format({
    "cenario_conservador": "{:.3f}", "cenario_base": "{:.3f}",
    "cenario_otimista": "{:.3f}", "demanda_mensal_base_milhoes": "{:.3f}"
}))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
historico_recente = df.tail(48)
ax.plot(historico_recente["data"], historico_recente["vendas_saar_milhoes"], color=COR_PRIMARIA, label="Histórico")
ax.plot(previsoes["data"], previsoes["cenario_base"], color=COR_SECUNDARIA, marker="o", linewidth=2.2, label="Previsão base")
ax.fill_between(previsoes["data"], previsoes["cenario_conservador"], previsoes["cenario_otimista"],
                 color=COR_SECUNDARIA, alpha=0.20, label="Faixa p10-p90 (bootstrap)")
ax.axvline(df["data"].max(), color="black", linestyle=":", linewidth=1)
ax.set_title(f"Previsão de demanda — {HORIZONTE_PREVISAO} meses ({melhor_modelo})", fontweight="bold")
ax.set_ylabel("Milhões de unidades (SAAR)")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Otimização do plano de produção

A previsão de mercado ainda não é um plano operacional. Nesta etapa, ela é traduzida para uma carteira fictícia de responsabilidade por meio de uma participação de mercado **puramente ilustrativa**. A programação linear decide quanto produzir em cada mês, respeitando capacidade e ponderando os custos de produção, estoque e falta.

| Parâmetro | Interpretação | Natureza |
|---|---|---|
| Participação de mercado | Fração da demanda total atribuída à carteira planejada | Hipótese didática editável |
| Capacidade mensal | Limite de produção de uma rede ou linha hipotética | Hipótese didática editável |
| Estoque inicial | Unidades disponíveis no início do plano | Hipótese didática editável |
| Custos | Penalidades econômicas para produção, estoque e ruptura | Hipótese didática editável |

Diferente da versão anterior, a decisão agora é avaliada em **três cenários de demanda** (conservador, base, otimista) e submetida a uma **análise de sensibilidade** de capacidade e participação de mercado — para explicitar o risco em torno das duas premissas mais incertas, em vez de apresentar um único número como se fosse certo.

> Em um projeto real, esses valores seriam substituídos por capacidade por planta, mix de modelos, custo por veículo, estoque físico, disponibilidade de peças e níveis de serviço acordados.

**Formulação:** para cada mês *t*, com produção *P<sub>t</sub>*, estoque final *I<sub>t</sub>* e demanda não atendida *B<sub>t</sub>*, minimiza-se o custo total sujeito ao balanço de estoque *I<sub>t</sub> − B<sub>t</sub> = I<sub>t−1</sub> − B<sub>t−1</sub> + P<sub>t</sub> − D<sub>t</sub>*, com *0 ≤ P<sub>t</sub> ≤ Capacidade* e variáveis não negativas. O custo de ruptura é maior que o de estocagem para priorizar nível de serviço.


In [ ]:
# Premissas didáticas e editáveis da otimização
PARTICIPACAO_ILUSTRATIVA = 0.08       # 8% do mercado total; NÃO representa a Stellantis
CAPACIDADE_MENSAL = 110_000           # veículos/mês de uma operação hipotética; cenário propositalmente restritivo
ESTOQUE_INICIAL = 15_000              # veículos
CUSTO_PRODUCAO = 25_000               # US$/veículo; apenas para demonstração
CUSTO_ESTOQUE = 350                   # US$/veículo/mês; apenas para demonstração
CUSTO_RUPTURA = 45_000                # US$/veículo não atendido; penalidade superior ao custo de produção

def converter_demanda_veiculos(cenario_milhoes_saar, participacao=PARTICIPACAO_ILUSTRATIVA):
    """Converte SAAR (milhões, anualizado) em unidades mensais de uma carteira hipotética."""
    return (cenario_milhoes_saar / 12 * 1_000_000 * participacao).round().astype(int)

plano = previsoes[["data", "cenario_conservador", "cenario_base", "cenario_otimista"]].copy()
plano["demanda_planejada_veiculos"] = converter_demanda_veiculos(plano["cenario_base"])
plano[["data", "demanda_planejada_veiculos"]]


In [ ]:
def resolver_plano_producao(demanda, capacidade, estoque_inicial,
                              custo_producao, custo_estoque, custo_ruptura, nome="plano"):
    """Programação linear: minimiza custo de produção + estoque + ruptura sob capacidade mensal."""
    periodos = list(range(len(demanda)))
    modelo_lp = pulp.LpProblem(f"Planejamento_{nome}", pulp.LpMinimize)

    producao = pulp.LpVariable.dicts("producao", periodos, lowBound=0, upBound=capacidade, cat="Continuous")
    estoque = pulp.LpVariable.dicts("estoque", periodos, lowBound=0, cat="Continuous")
    backlog = pulp.LpVariable.dicts("backlog", periodos, lowBound=0, cat="Continuous")

    modelo_lp += pulp.lpSum(
        custo_producao * producao[t] + custo_estoque * estoque[t] + custo_ruptura * backlog[t]
        for t in periodos
    )
    for t in periodos:
        demanda_t = float(demanda[t])
        inventario_anterior = estoque_inicial if t == 0 else estoque[t - 1]
        backlog_anterior = 0 if t == 0 else backlog[t - 1]
        modelo_lp += (
            estoque[t] - backlog[t] == inventario_anterior - backlog_anterior + producao[t] - demanda_t
        ), f"balanco_{t}"

    status = modelo_lp.solve(pulp.PULP_CBC_CMD(msg=False))
    if pulp.LpStatus[status] != "Optimal":
        raise RuntimeError(f"Otimização não encontrou solução ótima para '{nome}'. Status: {pulp.LpStatus[status]}")

    return {
        "status": pulp.LpStatus[status],
        "producao": [round(pulp.value(producao[t])) for t in periodos],
        "estoque": [round(pulp.value(estoque[t])) for t in periodos],
        "backlog": [round(pulp.value(backlog[t])) for t in periodos],
        "custo_total": pulp.value(modelo_lp.objective),
    }

print("Função de otimização definida.")


In [ ]:
# Plano recomendado no cenário base
resultado_base = resolver_plano_producao(
    plano["demanda_planejada_veiculos"].to_numpy(), CAPACIDADE_MENSAL, ESTOQUE_INICIAL,
    CUSTO_PRODUCAO, CUSTO_ESTOQUE, CUSTO_RUPTURA, nome="cenario_base"
)
plano["producao_recomendada"] = resultado_base["producao"]
plano["estoque_final"] = resultado_base["estoque"]
plano["demanda_pendente"] = resultado_base["backlog"]
plano["utilizacao_capacidade_pct"] = plano["producao_recomendada"] / CAPACIDADE_MENSAL * 100

print(f"Status da otimização: {resultado_base['status']}")
print(f"Custo total ilustrativo (cenário base): US$ {resultado_base['custo_total']:,.0f}")

display(plano[[
    "data", "demanda_planejada_veiculos", "producao_recomendada",
    "estoque_final", "demanda_pendente", "utilizacao_capacidade_pct"
]].style.format({"utilizacao_capacidade_pct": "{:.1f}%"}))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].bar(plano["data"], plano["demanda_planejada_veiculos"], width=18, color="#9dc3e6", label="Demanda planejada")
axes[0].plot(plano["data"], plano["producao_recomendada"], color=COR_SECUNDARIA, marker="o", linewidth=2.2, label="Produção recomendada")
axes[0].axhline(CAPACIDADE_MENSAL, color="black", linestyle="--", label="Capacidade")
axes[0].set_title("Plano de produção — cenário base", fontweight="bold")
axes[0].set_ylabel("Veículos")
axes[0].legend()

axes[1].plot(plano["data"], plano["estoque_final"], color=COR_PRIMARIA, marker="o", linewidth=2, label="Estoque final")
axes[1].plot(plano["data"], plano["demanda_pendente"], color=COR_ALERTA, marker="o", linestyle="--", linewidth=2, label="Demanda pendente")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Estoques e risco de ruptura", fontweight="bold")
axes[1].set_ylabel("Veículos")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Plano sob os três cenários de demanda (conservador, base, otimista)
comparacao_cenarios = []
for nome_cenario, coluna in [("Conservador", "cenario_conservador"), ("Base", "cenario_base"), ("Otimista", "cenario_otimista")]:
    demanda_cenario = converter_demanda_veiculos(plano[coluna])
    resultado = resolver_plano_producao(
        demanda_cenario.to_numpy(), CAPACIDADE_MENSAL, ESTOQUE_INICIAL,
        CUSTO_PRODUCAO, CUSTO_ESTOQUE, CUSTO_RUPTURA, nome=nome_cenario
    )
    comparacao_cenarios.append({
        "Cenário": nome_cenario,
        "Demanda total (veículos)": int(demanda_cenario.sum()),
        "Produção total (veículos)": int(sum(resultado["producao"])),
        "Utilização média (%)": np.mean(resultado["producao"]) / CAPACIDADE_MENSAL * 100,
        "Demanda pendente final": resultado["backlog"][-1],
        "Custo total (US$)": resultado["custo_total"],
    })

comparacao_cenarios = pd.DataFrame(comparacao_cenarios)
display(comparacao_cenarios.style.format({
    "Demanda total (veículos)": "{:,.0f}", "Produção total (veículos)": "{:,.0f}",
    "Utilização média (%)": "{:.1f}%", "Demanda pendente final": "{:,.0f}",
    "Custo total (US$)": "US$ {:,.0f}"
}))


In [ ]:
# Análise de sensibilidade: capacidade x participação de mercado (demanda no cenário base)
grade_capacidade = [round(CAPACIDADE_MENSAL * fator) for fator in (0.8, 0.9, 1.0, 1.1, 1.2)]
grade_participacao = [0.06, 0.08, 0.10]

matriz_backlog = pd.DataFrame(index=grade_capacidade, columns=grade_participacao, dtype=float)

for capacidade in grade_capacidade:
    for participacao in grade_participacao:
        demanda_grade = converter_demanda_veiculos(plano["cenario_base"], participacao=participacao)
        resultado_grade = resolver_plano_producao(
            demanda_grade.to_numpy(), capacidade, ESTOQUE_INICIAL,
            CUSTO_PRODUCAO, CUSTO_ESTOQUE, CUSTO_RUPTURA, nome=f"grade_{capacidade}_{participacao}"
        )
        matriz_backlog.loc[capacidade, participacao] = sum(resultado_grade["backlog"])

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(matriz_backlog, annot=True, fmt=",.0f", cmap="YlOrRd",
            cbar_kws={"label": "Demanda pendente acumulada (veículos)"}, ax=ax)
ax.set_title("Sensibilidade: demanda pendente sob capacidade e participação de mercado", fontweight="bold")
ax.set_xlabel("Participação de mercado hipotética")
ax.set_ylabel("Capacidade mensal (veículos)")
plt.tight_layout()
plt.show()

print(
    f"Leitura: cada célula mostra a demanda pendente acumulada no horizonte de "
    f"{HORIZONTE_PREVISAO} meses sob uma combinação de capacidade e participação de mercado. "
    "O mapa evidencia a partir de que ponto a capacidade deixa de ser suficiente, sem "
    "depender de um único cenário fixo."
)


## 5. Conclusões executivas

O notebook deve ser interpretado como uma demonstração de raciocínio quantitativo aplicado à indústria automotiva. Na entrevista, o foco não é afirmar que o número previsto é uma decisão real da empresa, mas explicar um processo disciplinado: validar dados, testar modelos sem vazamento temporal em múltiplas janelas, diagnosticar os resíduos, quantificar a incerteza sem depender de suposições frágeis, e converter previsão em decisão sob restrições — inclusive testando a sensibilidade dessa decisão.

| Mensagem | Como defender na entrevista |
|---|---|
| A validação usa 4 janelas walk-forward, não um único corte | "Um único split pode ser sorte ou azar; a validação cruzada temporal mostra que o modelo é consistente ao longo de várias janelas." |
| Três modelos são comparados, incluindo uma referência simples | "Complexidade só se justifica se reduzir o erro fora da amostra de forma consistente." |
| Os resíduos do modelo vencedor foram testados quanto a autocorrelação | "Se sobrasse padrão previsível no erro, o modelo estaria mal especificado; o teste de Ljung-Box e a ACF dos resíduos apoiam essa checagem." |
| A incerteza vem de bootstrap dos erros reais, não de uma suposição normal | "Não assumi normalidade; reamostrei os erros observados no próprio backtest." |
| A decisão foi avaliada em três cenários e sob um mapa de sensibilidade | "Não apostei em um único número; mostrei como a decisão muda com capacidade e participação de mercado." |
| O próximo passo é granularidade | "Com dados internos, eu modelaria por modelo, região, planta e restrições de fornecedores." |

### Exportação

A tabela abaixo é salva em CSV para permitir auditoria, compartilhamento ou uso em um painel posterior.


In [ ]:
# Exportação e resumo final
arquivo_saida = "plano_producao_automotivo_6_meses.csv"
plano.to_csv(arquivo_saida, index=False)

resumo_final = pd.DataFrame({
    "Indicador": [
        "Modelo selecionado",
        f"MAPE médio (walk-forward, {N_DOBRAS_CV} dobras)",
        "Ljung-Box (lag 12) p-valor",
        "Demanda planejada total (cenário base)",
        "Produção recomendada total (cenário base)",
        "Utilização média da capacidade (cenário base)",
        "Demanda pendente final (cenário base)",
        "Demanda pendente final (cenário otimista)",
    ],
    "Resultado": [
        melhor_modelo,
        f"{resumo_cv.loc[0, 'mape_medio']:.2f}% (± {resumo_cv.loc[0, 'mape_desvio']:.2f} p.p.)",
        f"{teste_ljung_box.loc[12, 'lb_pvalue']:.3f}",
        f"{plano['demanda_planejada_veiculos'].sum():,.0f} veículos",
        f"{plano['producao_recomendada'].sum():,.0f} veículos",
        f"{plano['utilizacao_capacidade_pct'].mean():.1f}%",
        f"{plano['demanda_pendente'].iloc[-1]:,.0f} veículos",
        f"{comparacao_cenarios.loc[comparacao_cenarios['Cenário'] == 'Otimista', 'Demanda pendente final'].iloc[0]:,.0f} veículos",
    ]
})
display(resumo_final)
print(f"Arquivo gerado: {arquivo_saida}")


## Referências

[1] [FRED — Total Vehicle Sales (TOTALSA)](https://fred.stlouisfed.org/series/TOTALSA). Série mensal de vendas totais de veículos, em milhões de unidades e taxa anual ajustada sazonalmente; fonte original: U.S. Bureau of Economic Analysis.

[2] [Federal Reserve — Seasonal Factors for Motor Vehicle Sales](https://www.federalreserve.gov/releases/g17/mv_sales_sf.htm). Documentação dos fatores sazonais para vendas de veículos e do uso de X-13 ARIMA na estimação.

[3] [Bureau of Transportation Statistics — Auto Sales](https://catalog.data.gov/dataset/auto-sales). Catálogo de dados públicos sobre vendas de automóveis com referência à divulgação pelo BEA.

[4] Cleveland, R. B.; Cleveland, W. S.; McRae, J. E.; Terpenning, I. (1990). "STL: A Seasonal-Trend Decomposition Procedure Based on Loess." *Journal of Official Statistics*. Método usado na decomposição da série.

[5] Ljung, G. M.; Box, G. E. P. (1978). "On a Measure of Lack of Fit in Time Series Models." *Biometrika*. Base do teste de autocorrelação dos resíduos.

[6] Efron, B. (1979). "Bootstrap Methods: Another Look at the Jackknife." *Annals of Statistics*. Base do método de reamostragem usado para estimar a incerteza da previsão.
